# Asset Statistical Pipeline

Parameterized diagnostic + alpha pipeline for any single asset or spread.

**Pipeline:**
```
Config → Load → Stationarity → Drift → ACF/PACF → FFT → Hurst → VR → AR
       → Features → Alpha Score → Backtest → Risk/Inventory → Sensitivity
```

## 0. Configuration — edit here only

In [ ]:
# ── Asset ──────────────────────────────────────────────────────────────────
PRODUCT    = "KELP"        # product column name in CSV
DATA_DIR   = "../imc-prosperity-3-backtester/prosperity3bt/resources/round2"
FILE_PREFIX = "prices_round_2_day_"   # filename stem before the day number
DAY_RANGE  = [-1, 0, 1]              # days to concatenate

# ── Spread mode ────────────────────────────────────────────────────────────
# Set IS_SPREAD=True to analyse basket - weighted synthetic instead
IS_SPREAD     = False
BASKET_PRODUCT  = "PICNIC_BASKET1"
SYNTH_WEIGHTS   = {"CROISSANTS": 6, "JAMS": 3, "DJEMBES": 1}
SPREAD_LABEL    = "6×CROISSANTS + 3×JAMS + 1×DJEMBES"

# ── Statistical diagnostics ────────────────────────────────────────────────
N_WINDOW     = 20     # rolling window for features
N_LAGS_ACF  = 40     # lags shown in ACF/PACF
LB_LAGS     = [1, 2, 3, 5, 10, 20]   # Ljung-Box lag horizons
HURST_NMAX  = 100    # max sub-series length for R/S
VR_LAGS     = [2, 4, 8, 16]          # lags for variance-ratio test
AR_MAX_P    = 10     # max AR order to search
FFT_TOP_N   = 5      # top N periodicities to label

# ── Strategy ───────────────────────────────────────────────────────────────
POS_LIMIT       = 60
UNIT_SIZE       = 1
ENTRY_THRESHOLD = 0.35
EXIT_THRESH     = 0.05
SIG_WEIGHTS     = {"sig_zscore": 0.40, "sig_rsi": 0.25,
                   "sig_bb":     0.25, "sig_macd": 0.10}

# ── Risk / inventory ───────────────────────────────────────────────────────
SOFT_LIMIT  = 40      # start reducing aggressiveness
MTM_PAIN    = -500    # force-close threshold on unrealised PnL
STALE_BARS  = 200     # bars before stale-position penalty kicks in
ROLL_W      = 200     # rolling Sharpe window

## 1. Imports & Helpers

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf, acovf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.ar_model import AutoReg

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.titlesize'] = 13

# ── Indicator library ──────────────────────────────────────────────────────
def sma(s, n):       return s.rolling(n).mean()
def ema(s, n):       return s.ewm(span=n, adjust=False).mean()

def rsi(s, n=14):
    d = s.diff()
    g = d.clip(lower=0).rolling(n).mean()
    l = (-d.clip(upper=0)).rolling(n).mean()
    return 100 - 100 / (1 + g / l.replace(0, np.nan))

def macd(s, fast=12, slow=26, sig=9):
    line = ema(s, fast) - ema(s, slow)
    signal = ema(line, sig)
    return line, signal, line - signal

def bollinger(s, n=20, k=2):
    mid = sma(s, n); std = s.rolling(n).std()
    return mid - k*std, mid, mid + k*std

def zscore(s, n=20):
    return (s - s.rolling(n).mean()) / s.rolling(n).std().replace(0, np.nan)

def momentum(s, n=10): return s - s.shift(n)

print("Imports OK")

## 2. Load Data

In [ ]:
dfs = []
for day in DAY_RANGE:
    path = f"{DATA_DIR}/{FILE_PREFIX}{day}.csv"
    df = pd.read_csv(path, sep=";")
    df["day"] = day
    dfs.append(df)
prices = pd.concat(dfs, ignore_index=True)
prices["t"] = prices["day"] * 1_000_000 + prices["timestamp"]

def get_mid(product):
    sub = prices[prices["product"] == product][["t", "mid_price"]].set_index("t")
    return sub["mid_price"]

if IS_SPREAD:
    basket = get_mid(BASKET_PRODUCT)
    components = {p: get_mid(p) for p in SYNTH_WEIGHTS}
    common = basket.index
    for s in components.values(): common = common.intersection(s.index)
    basket = basket.loc[common]
    synthetic = sum(w * components[p].loc[common] for p, w in SYNTH_WEIGHTS.items())
    series = basket - synthetic
    series.name = f"{BASKET_PRODUCT} - Synthetic"
    price_series = basket   # for raw price plots
    print(f"Spread mean: {series.mean():.2f}  std: {series.std():.2f}")
else:
    series = get_mid(PRODUCT)
    price_series = series

series = series.dropna()
log_ret = np.log(price_series / price_series.shift(1)).dropna()

print(f"Loaded {len(series):,} observations")
print(f"Price range: {series.min():.2f} – {series.max():.2f}")
print(f"Mean: {series.mean():.4f}  Std: {series.std():.4f}")

## 3. Price Series Overview

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(series.values, color='steelblue', linewidth=0.8, label=series.name or PRODUCT)
axes[0].plot(sma(series, N_WINDOW).values, color='darkorange', linewidth=1.2,
             linestyle='--', label=f'SMA{N_WINDOW}')
axes[0].set_title('Price / Spread — Raw Series + SMA')
axes[0].legend()

vol = series.rolling(N_WINDOW).std()
axes[1].plot(vol.values, color='crimson', linewidth=0.8)
axes[1].set_title(f'Rolling {N_WINDOW}-bar Volatility')

plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(log_ret.values, color='steelblue', linewidth=0.6, alpha=0.7)
ax.axhline(0, color='gray', linewidth=0.5, linestyle=':')
ax.set_title('Log Returns')
plt.tight_layout(); plt.show()

## 4. Stationarity Tests — ADF + KPSS

| Result | ADF p | KPSS p | Interpretation |
|--------|-------|--------|----------------|
| Stationary I(0) | < 0.05 | > 0.05 | ✅ mean-reversion OK |
| Unit root I(1) | > 0.05 | < 0.05 | ❌ difference first |
| Ambiguous | both fail | — | check visually |

In [ ]:
def run_adf_kpss(s, label='series'):
    adf_stat, adf_p, _, _, adf_cv, _ = adfuller(s.dropna(), autolag='AIC')
    try:
        kpss_stat, kpss_p, _, kpss_cv = kpss(s.dropna(), regression='c', nlags='auto')
    except Exception:
        kpss_stat, kpss_p = np.nan, np.nan

    stationary_adf  = adf_p  < 0.05
    stationary_kpss = kpss_p > 0.05

    print(f"\n{'═'*55}")
    print(f"  Stationarity Tests — {label}")
    print(f"{'═'*55}")
    print(f"  ADF  stat={adf_stat:>9.4f}  p={adf_p:.4f}  "
          f"{'[STATIONARY ✓]' if stationary_adf else '[UNIT ROOT ✗]'}")
    print(f"  KPSS stat={kpss_stat:>9.4f}  p={kpss_p:.4f}  "
          f"{'[STATIONARY ✓]' if stationary_kpss else '[NON-STAT  ✗]'}")

    if stationary_adf and stationary_kpss:
        verdict = "I(0) — stationary  ✅  Mean-reversion strategies applicable"
    elif not stationary_adf and not stationary_kpss:
        verdict = "I(1) — unit root   ❌  Difference the series before modelling"
    else:
        verdict = "Ambiguous — inspect visually or test on differences"
    print(f"\n  Verdict: {verdict}")
    return adf_p, kpss_p

label = f"{BASKET_PRODUCT} - Synthetic" if IS_SPREAD else PRODUCT
adf_p_level, kpss_p_level = run_adf_kpss(series,   label=f'{label} (levels)')
adf_p_ret,   kpss_p_ret   = run_adf_kpss(log_ret,  label=f'{label} (log-returns)')

## 5. Drift Detection — OLS Trend

In [ ]:
t_idx = np.arange(len(series))
X = sm.add_constant(t_idx)
ols = sm.OLS(series.values, X).fit()
slope, slope_pval = ols.params[1], ols.pvalues[1]

print(f"OLS drift: slope={slope:.6f}, p={slope_pval:.4f}, R²={ols.rsquared:.4f}")
if slope_pval < 0.05:
    print(f"  → Significant {'upward' if slope > 0 else 'downward'} drift detected ⚠")
else:
    print("  → No significant drift — zero-drift assumption holds ✅")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(series.values, color='steelblue', linewidth=0.8, label='Series', alpha=0.8)
ax.plot(ols.fittedvalues, color='red', linewidth=1.5, linestyle='--',
        label=f'OLS trend (slope={slope:.5f}, p={slope_pval:.3f})')
ax.set_title('Drift Detection — OLS Fit')
ax.legend(); plt.tight_layout(); plt.show()

## 6. Autocorrelation — ACF / PACF / Ljung-Box

In [ ]:
ret_arr = log_ret.values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sm.graphics.tsa.plot_acf( ret_arr, lags=N_LAGS_ACF, ax=axes[0], title='ACF  (log returns)',  alpha=0.05)
sm.graphics.tsa.plot_pacf(ret_arr, lags=N_LAGS_ACF, ax=axes[1], title='PACF (log returns)', alpha=0.05)
plt.tight_layout(); plt.show()

lb = acorr_ljungbox(ret_arr, lags=LB_LAGS, return_df=True)
print("\nLjung-Box test on log-returns:")
print(lb.rename(columns={'lb_stat':'Q-stat','lb_pvalue':'p-value'}).round(4).to_string())
sig_lbs = lb[lb['lb_pvalue'] < 0.05]
if len(sig_lbs):
    print(f"  → Significant autocorrelation at lags: {list(sig_lbs.index)} ⚠")
else:
    print("  → No significant autocorrelation — returns are close to i.i.d. ✅")

## 7. Cyclicality — FFT Periodogram

In [ ]:
detrended = series.values - ols.fittedvalues
N = len(detrended)
fft_vals  = np.fft.rfft(detrended)
freqs     = np.fft.rfftfreq(N)
power     = np.abs(fft_vals)**2

# exclude DC component
pos = freqs > 0
freqs_pos, power_pos = freqs[pos], power[pos]
periods = 1.0 / freqs_pos

top_idx = np.argsort(power_pos)[-FFT_TOP_N:][::-1]

fig, ax = plt.subplots(figsize=(14, 5))
ax.loglog(periods, power_pos, color='steelblue', linewidth=0.8)
for i in top_idx:
    ax.axvline(periods[i], color='red', linewidth=0.8, linestyle='--', alpha=0.7)
    ax.text(periods[i], power_pos[i], f' {periods[i]:.0f}b', fontsize=8,
            color='red', va='bottom')
ax.set_xlabel('Period (bars)'); ax.set_ylabel('Power')
ax.set_title('FFT Periodogram (detrended series)')
plt.tight_layout(); plt.show()

print(f"Top {FFT_TOP_N} dominant periods (bars): "
      + ", ".join(f"{periods[i]:.0f}" for i in top_idx))

## 8. Hurst Exponent — R/S Analysis

| H | Regime |
|---|--------|
| < 0.5 | Mean-reverting |
| ≈ 0.5 | Random walk |
| > 0.5 | Trending / persistent |

In [ ]:
def hurst_rs(ts, n_max=None):
    ts = np.asarray(ts, dtype=float)
    ts = ts[~np.isnan(ts)]
    N  = len(ts)
    if n_max is None: n_max = N // 2
    lags, rs_vals = [], []
    for n in range(10, min(n_max, N//2)+1, max(1, (min(n_max, N//2)-10)//50)):
        chunks = [ts[i:i+n] for i in range(0, N-n+1, n)]
        rs_chunk = []
        for chunk in chunks:
            mean  = chunk.mean()
            dev   = np.cumsum(chunk - mean)
            R     = dev.max() - dev.min()
            S     = chunk.std(ddof=1)
            if S > 0: rs_chunk.append(R / S)
        if rs_chunk:
            lags.append(n); rs_vals.append(np.mean(rs_chunk))
    lags, rs_vals = np.array(lags), np.array(rs_vals)
    log_n, log_rs = np.log(lags), np.log(rs_vals)
    H, _ = np.polyfit(log_n, log_rs, 1)
    ss_res = np.sum((log_rs - np.polyval([H, _], log_n))**2)
    ss_tot = np.sum((log_rs - log_rs.mean())**2)
    r2 = 1 - ss_res/ss_tot if ss_tot > 0 else np.nan
    return H, r2, lags, rs_vals

H, r2, lags, rs_vals = hurst_rs(series.values, n_max=HURST_NMAX)
regime = "Mean-reverting ✅" if H < 0.45 else ("Random walk" if H < 0.55 else "Trending ⚠")
print(f"Hurst exponent H = {H:.4f}  (R²={r2:.4f})")
print(f"Regime: {regime}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.loglog(lags, rs_vals, 'o', color='steelblue', markersize=4, label='R/S data')
fit_line = np.exp(np.polyval([H, np.log(rs_vals[0]) - H*np.log(lags[0])], np.log(lags)))
ax.loglog(lags, fit_line, '--', color='red', label=f'H={H:.3f}  R²={r2:.3f}')
ax.set_xlabel('Lag n'); ax.set_ylabel('E[R/S]')
ax.set_title('Hurst R/S Analysis')
ax.legend(); plt.tight_layout(); plt.show()

## 9. Variance Ratio Test — Lo-MacKinlay

In [ ]:
def variance_ratio(ret, k):
    """Heteroskedasticity-robust VR test (Lo-MacKinlay 1988)."""
    T  = len(ret)
    mu = ret.mean()
    # k-period returns
    ret_k = np.array([ret[i:i+k].sum() for i in range(T-k+1)])
    var1  = ((ret - mu)**2).sum() / (T - 1)
    vark  = ((ret_k - k*mu)**2).sum() / (len(ret_k) - 1) / k
    vr    = vark / var1
    # heteroskedasticity-robust z-stat
    delta = np.array([
        ((ret[j]**2) * (ret[j+1:j+k].sum())**2
         for j in range(T-k)) if False else 0
    ])
    # simplified z-stat (homoskedastic version)
    z = (vr - 1) / np.sqrt(2*(2*k - 1)*(k-1) / (3*k*T))
    return vr, z

ret_arr_vr = log_ret.values
print(f"{'Lag k':>8} {'VR':>8} {'z-stat':>10} {'Regime':>20}")
print('-' * 50)
vr_results = []
for k in VR_LAGS:
    vr, z = variance_ratio(ret_arr_vr, k)
    regime_vr = "Mean-reverting" if vr < 0.95 else ("Random walk" if vr < 1.05 else "Trending")
    print(f"{k:>8} {vr:>8.4f} {z:>10.4f} {regime_vr:>20}")
    vr_results.append((k, vr, z))

fig, ax = plt.subplots(figsize=(10, 5))
ks = [r[0] for r in vr_results]
vrs = [r[1] for r in vr_results]
ax.plot(ks, vrs, 'o-', color='steelblue', label='VR')
ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.8, label='Random walk (VR=1)')
ax.fill_between(ks, [0.95]*len(ks), [1.05]*len(ks), alpha=0.1, color='gray')
ax.set_xlabel('Lag k'); ax.set_ylabel('Variance Ratio')
ax.set_title('Variance Ratio Profile')
ax.legend(); plt.tight_layout(); plt.show()

## 10. AR Model Selection — AIC / BIC

In [ ]:
aic_vals, bic_vals = [], []
for p in range(1, AR_MAX_P + 1):
    try:
        m = AutoReg(series.values, lags=p, trend='c').fit()
        aic_vals.append(m.aic); bic_vals.append(m.bic)
    except Exception:
        aic_vals.append(np.nan); bic_vals.append(np.nan)

best_p_aic = np.nanargmin(aic_vals) + 1
best_p_bic = np.nanargmin(bic_vals) + 1
print(f"Best AR order by AIC: {best_p_aic}")
print(f"Best AR order by BIC: {best_p_bic}")

best_p = best_p_bic  # BIC is more parsimonious
ar_model = AutoReg(series.values, lags=best_p, trend='c').fit()
print(f"\nAR({best_p}) coefficients (lag 1 to {best_p}):")
for i, c in enumerate(ar_model.params[1:best_p+1], 1):
    print(f"  φ_{i} = {c:.6f}")
print(f"  Intercept = {ar_model.params[0]:.4f}")

if len(ar_model.params) > 1 and abs(ar_model.params[1]) > 0.01:
    sign = ar_model.params[1]
    print(f"\n  AR(1) coeff = {sign:.4f} → "
          f"{'mean-reverting tendency ✅' if sign < 0 else 'persistence / trending ⚠'}")

fig, ax = plt.subplots(figsize=(10, 4))
lags_range = range(1, AR_MAX_P + 1)
ax.plot(lags_range, aic_vals, 'o-', color='steelblue', label='AIC')
ax.plot(lags_range, bic_vals, 's-', color='darkorange', label='BIC')
ax.axvline(best_p_bic, color='red', linestyle='--', linewidth=0.8,
           label=f'Best BIC lag={best_p_bic}')
ax.set_xlabel('AR lag order p'); ax.set_ylabel('Information Criterion')
ax.set_title('AR Model Selection — AIC / BIC')
ax.legend(); plt.tight_layout(); plt.show()

## 11. Statistical Summary & Strategy Implications

In [ ]:
rows = [
    ("ADF (levels)",        f"p={adf_p_level:.4f}",
     "Stationary ✅" if adf_p_level < 0.05 else "Unit root ❌"),
    ("KPSS (levels)",       f"p={kpss_p_level:.4f}",
     "Stationary ✅" if kpss_p_level > 0.05 else "Non-stationary ❌"),
    ("ADF (log-ret)",       f"p={adf_p_ret:.4f}",
     "Stationary ✅" if adf_p_ret < 0.05 else "Unit root ❌"),
    ("KPSS (log-ret)",      f"p={kpss_p_ret:.4f}",
     "Stationary ✅" if kpss_p_ret > 0.05 else "Non-stationary ❌"),
    ("Drift (OLS)",         f"slope_p={slope_pval:.4f}",
     "No drift ✅" if slope_pval >= 0.05 else f"Drift {'↑' if slope > 0 else '↓'} ⚠"),
    ("Hurst exponent",      f"H={H:.4f}",
     "Mean-reverting ✅" if H < 0.45 else ("Random walk" if H < 0.55 else "Trending ⚠")),
    ("AR(1) coefficient",   f"φ₁={ar_model.params[1] if best_p>=1 else np.nan:.4f}",
     "Mean-revert tendency ✅" if best_p >= 1 and ar_model.params[1] < 0 else "Persistent ⚠"),
    ("VR (k=2)",            f"VR={vr_results[0][1]:.4f}",
     "Mean-reverting ✅" if vr_results[0][1] < 0.95 else
     ("Random walk" if vr_results[0][1] < 1.05 else "Trending ⚠")),
]

summary = pd.DataFrame(rows, columns=['Test', 'Value', 'Verdict'])
print(summary.to_string(index=False))

n_good = sum(1 for _, _, v in rows if '✅' in v)
print(f"\n{'─'*55}")
print(f"  Strategy fitness: {n_good}/{len(rows)} indicators favour mean-reversion")
if n_good >= 5:
    print("  → STRONG candidate for mean-reversion / stat-arb strategy ✅")
elif n_good >= 3:
    print("  → MODERATE — consider risk controls, regime gating ⚠")
else:
    print("  → WEAK — series may be trending; reconsider approach ❌")

## 12. Feature Engineering

In [ ]:
feat = pd.DataFrame(index=series.index)
feat['series']     = series
feat['sma']        = sma(series, N_WINDOW)
feat['ema']        = ema(series, N_WINDOW)
feat['zscore']     = zscore(series, N_WINDOW)
feat['rsi']        = rsi(series, 14)
feat['bb_lo'], feat['bb_mid'], feat['bb_hi'] = bollinger(series, N_WINDOW, 2)
feat['macd_line'], feat['macd_sig'], feat['macd_hist'] = macd(series)
feat['momentum']   = momentum(series, 10)
feat.dropna(inplace=True)
print(feat.describe().round(4))

## 13. Strategy Dashboard

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)

ax = axes[0]
ax.plot(feat['series'].values, color='steelblue', linewidth=0.8, label='Series')
ax.plot(feat['bb_mid'].values, color='gray', linestyle='--', linewidth=0.8, label=f'SMA{N_WINDOW}')
ax.fill_between(range(len(feat)), feat['bb_lo'].values, feat['bb_hi'].values,
                alpha=0.15, color='steelblue', label='BB ±2σ')
ax.axhline(feat['series'].mean(), color='black', linewidth=0.5, linestyle=':')
ax.set_title('Series with Bollinger Bands'); ax.legend(loc='upper right', fontsize=9)

ax = axes[1]
ax.plot(feat['zscore'].values, color='purple', linewidth=0.8)
ax.axhline(2,  color='red',   linestyle='--', linewidth=0.8)
ax.axhline(-2, color='green', linestyle='--', linewidth=0.8)
ax.axhline(0,  color='gray',  linestyle=':',  linewidth=0.5)
ax.set_title(f'Z-score ({N_WINDOW}-bar rolling)')

ax = axes[2]
ax.plot(feat['rsi'].values, color='darkorange', linewidth=0.8)
ax.axhline(70, color='red',   linestyle='--', linewidth=0.8)
ax.axhline(30, color='green', linestyle='--', linewidth=0.8)
ax.set_ylim(0, 100); ax.set_title('RSI(14)')

ax = axes[3]
ax.plot(feat['macd_line'].values, color='steelblue',  linewidth=0.8, label='MACD')
ax.plot(feat['macd_sig'].values,  color='darkorange', linewidth=0.8, label='Signal')
colors = ['green' if v >= 0 else 'red' for v in feat['macd_hist'].values]
ax.bar(range(len(feat)), feat['macd_hist'].values, color=colors, alpha=0.4, label='Histogram')
ax.axhline(0, color='gray', linestyle=':', linewidth=0.5)
ax.set_title('MACD'); ax.legend(loc='upper right', fontsize=9)

plt.tight_layout(); plt.show()

## 14. Multi-Signal Alpha Score

In [ ]:
# Positive composite → series expensive → short
feat['sig_zscore'] = feat['zscore'].clip(-3, 3) / 3
feat['sig_rsi']    = (feat['rsi'] - 50) / 50
feat['sig_bb']     = ((feat['series'] - feat['bb_mid']) /
                       (feat['bb_hi'] - feat['bb_mid']).replace(0, np.nan)).clip(-1, 1)
feat['sig_macd']   = np.sign(feat['macd_hist']).fillna(0)

feat['composite'] = sum(feat[k] * v for k, v in SIG_WEIGHTS.items())

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(feat['series'].values, linewidth=0.8, color='steelblue')
axes[0].set_title('Series')
axes[1].plot(feat['composite'].values, linewidth=0.8, color='navy')
axes[1].axhline( ENTRY_THRESHOLD, color='red',   linestyle='--', linewidth=0.8,
                 label=f'+{ENTRY_THRESHOLD} (short)')
axes[1].axhline(-ENTRY_THRESHOLD, color='green', linestyle='--', linewidth=0.8,
                 label=f'-{ENTRY_THRESHOLD} (long)')
axes[1].axhline(0, color='gray', linestyle=':', linewidth=0.5)
axes[1].set_title('Composite Alpha Score')
axes[1].legend(); plt.tight_layout(); plt.show()

print(f"Short signals: {(feat['composite'] >  ENTRY_THRESHOLD).sum()}")
print(f"Long  signals: {(feat['composite'] < -ENTRY_THRESHOLD).sum()}")

## 15. Signal Quality Analysis

In [ ]:
FWDS = [1, 5, 20]
rows_sq = []
for sig_col in SIG_WEIGHTS:
    direction = np.sign(feat[sig_col])
    for fwd in FWDS:
        fwd_ret     = feat['series'].shift(-fwd) - feat['series']
        aligned_ret = (direction * (-fwd_ret)).dropna()
        hit_rate    = (aligned_ret > 0).mean()
        rows_sq.append({'signal': sig_col, 'horizon': fwd,
                        'hit_rate': f'{hit_rate:.1%}',
                        'avg_pnl': f'{aligned_ret.mean():.4f}'})
print(pd.DataFrame(rows_sq).to_string(index=False))

## 16. Backtest Simulation

In [ ]:
positions = []
pos = 0
for _, row in feat.iterrows():
    sc = row['composite']
    if   sc >  ENTRY_THRESHOLD and pos > -POS_LIMIT: pos -= UNIT_SIZE
    elif sc < -ENTRY_THRESHOLD and pos <  POS_LIMIT: pos += UNIT_SIZE
    elif abs(sc) < EXIT_THRESH: pos = 0
    positions.append(pos)

positions  = np.array(positions)
series_arr = feat['series'].values
pnl        = np.diff(series_arr, prepend=series_arr[0]) * positions
cum_pnl    = np.cumsum(pnl)
sharpe     = pnl.mean() / (pnl.std() + 1e-9) * np.sqrt(len(pnl))
max_dd     = (cum_pnl - np.maximum.accumulate(cum_pnl)).min()

print(f"Total PnL:   {cum_pnl[-1]:>12.2f}")
print(f"Sharpe:      {sharpe:>12.3f}")
print(f"Max Drawdown:{max_dd:>12.2f}")
print(f"Avg |pos|:   {np.abs(positions).mean():>12.2f}")

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
axes[0].plot(cum_pnl, color='steelblue', linewidth=1.2, label='Strategy PnL')
axes[0].axhline(0, color='gray', linestyle=':', linewidth=0.5)
axes[0].set_title('Cumulative PnL'); axes[0].legend()
axes[1].plot(series_arr, color='steelblue', linewidth=0.8)
axes[1].set_title('Series')
axes[2].plot(positions, color='purple', linewidth=0.8)
axes[2].axhline(0, color='gray', linestyle=':', linewidth=0.5)
axes[2].set_title('Position')
plt.tight_layout(); plt.show()

## 17. Inventory-Aware Backtest

In [ ]:
STATE_BINS  = [0, 0.3, 0.6, 0.85, 1.01]
STATE_NAMES = ['Flat', 'Light', 'Moderate', 'Heavy']

def inv_state(pos): return STATE_NAMES[np.searchsorted(STATE_BINS, abs(pos)/POS_LIMIT, side='right') - 1]

pos2, positions2, entry_bar, entry_price = 0, [], None, None
pnl2_arr, states = [], []

for bar, (idx, row) in enumerate(feat.iterrows()):
    sc    = row['composite']
    price = row['series']
    abs_pos = abs(pos2)

    # MTM exit
    if entry_price is not None:
        mtm = (price - entry_price) * pos2
        if mtm < MTM_PAIN:
            pos2 = 0; entry_price = None; entry_bar = None

    # Stale position urgency
    urgency = 1.0
    if entry_bar is not None and (bar - entry_bar) > STALE_BARS:
        urgency = 1.5 + (bar - entry_bar - STALE_BARS) / STALE_BARS

    # Position limit scaling
    size_mul = max(0.0, 1.0 - (abs_pos - SOFT_LIMIT) / (POS_LIMIT - SOFT_LIMIT)) if abs_pos > SOFT_LIMIT else 1.0
    adj_unit = max(1, int(UNIT_SIZE * size_mul * urgency))

    if sc >  ENTRY_THRESHOLD and pos2 > -POS_LIMIT:
        pos2 -= adj_unit
        if entry_price is None: entry_price, entry_bar = price, bar
    elif sc < -ENTRY_THRESHOLD and pos2 <  POS_LIMIT:
        pos2 += adj_unit
        if entry_price is None: entry_price, entry_bar = price, bar
    elif abs(sc) < EXIT_THRESH:
        pos2 = 0; entry_price = None; entry_bar = None

    pos2 = np.clip(pos2, -POS_LIMIT, POS_LIMIT)
    positions2.append(pos2)
    states.append(inv_state(pos2))

positions2 = np.array(positions2)
pnl2 = np.diff(series_arr, prepend=series_arr[0]) * positions2
cum_pnl2 = np.cumsum(pnl2)
sharpe2  = pnl2.mean() / (pnl2.std() + 1e-9) * np.sqrt(len(pnl2))
max_dd2  = (cum_pnl2 - np.maximum.accumulate(cum_pnl2)).min()

print(f"Inventory-aware | PnL: {cum_pnl2[-1]:.2f}  Sharpe: {sharpe2:.3f}  MaxDD: {max_dd2:.2f}")
print(f"Baseline        | PnL: {cum_pnl[-1]:.2f}  Sharpe: {sharpe:.3f}  MaxDD: {max_dd:.2f}")

# ── Plot ──
state_colors = {'Flat': 'steelblue', 'Light': 'green', 'Moderate': 'orange', 'Heavy': 'red'}
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
axes[0].plot(cum_pnl,  color='gray',      linewidth=0.8, label='Baseline', alpha=0.6)
axes[0].plot(cum_pnl2, color='steelblue', linewidth=1.2, label='Inventory-aware')
axes[0].axhline(0, color='gray', linestyle=':', linewidth=0.5)
axes[0].set_title('PnL Comparison'); axes[0].legend()
axes[1].plot(series_arr, color='steelblue', linewidth=0.8)
axes[1].set_title('Series')
for i, (p, st) in enumerate(zip(positions2, states)):
    axes[2].bar(i, p, color=state_colors.get(st, 'gray'), alpha=0.5, width=1)
axes[2].axhline(0, color='gray', linestyle=':', linewidth=0.5)
axes[2].set_title('Position (coloured by inventory state)')
plt.tight_layout(); plt.show()

# ── PnL by state ──
pnl2_s = pd.Series(pnl2)
for st in STATE_NAMES:
    mask = np.array(states) == st
    p = pnl2_s[mask]
    print(f"  {st:10s} | n={mask.sum():5d}  PnL={p.sum():8.1f}  "
          f"Sharpe={p.mean()/(p.std()+1e-9)*np.sqrt(max(1,mask.sum())):.2f}")

## 18. Rolling Sharpe

In [ ]:
pnl_s = pd.Series(pnl2)
rolling_sharpe = (pnl_s.rolling(ROLL_W).mean() /
                  (pnl_s.rolling(ROLL_W).std() + 1e-9)) * np.sqrt(ROLL_W)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(rolling_sharpe.values, color='steelblue', linewidth=0.8)
ax.axhline(0, color='gray',  linestyle=':', linewidth=0.5)
ax.axhline(1, color='green', linestyle='--', linewidth=0.8, alpha=0.7, label='Sharpe=1')
ax.set_title(f'Rolling Sharpe ({ROLL_W}-bar)'); ax.set_ylabel('Sharpe')
ax.legend(); plt.tight_layout(); plt.show()

## 19. Parameter Sensitivity — Entry Threshold Sweep

In [ ]:
thresholds = np.arange(0.10, 0.81, 0.05)
sharpe_vals, pnl_vals = [], []

for thresh in thresholds:
    p_t, p_arr = 0, []
    for _, row in feat.iterrows():
        sc = row['composite']
        if   sc >  thresh and p_t > -POS_LIMIT: p_t -= UNIT_SIZE
        elif sc < -thresh and p_t <  POS_LIMIT: p_t += UNIT_SIZE
        elif abs(sc) < EXIT_THRESH: p_t = 0
        p_arr.append(p_t)
    pnl_t = np.diff(feat['series'].values, prepend=feat['series'].values[0]) * np.array(p_arr)
    sharpe_vals.append(pnl_t.mean() / (pnl_t.std() + 1e-9) * np.sqrt(len(pnl_t)))
    pnl_vals.append(pnl_t.sum())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(thresholds, sharpe_vals, 'o-', color='steelblue')
axes[0].axvline(ENTRY_THRESHOLD, color='red', linestyle='--',
                label=f'Current={ENTRY_THRESHOLD}')
axes[0].set_xlabel('Entry Threshold'); axes[0].set_ylabel('Sharpe')
axes[0].set_title('Sharpe vs Entry Threshold'); axes[0].legend()
axes[1].plot(thresholds, pnl_vals, 'o-', color='darkorange')
axes[1].axvline(ENTRY_THRESHOLD, color='red', linestyle='--',
                label=f'Current={ENTRY_THRESHOLD}')
axes[1].set_xlabel('Entry Threshold'); axes[1].set_ylabel('Total PnL')
axes[1].set_title('Total PnL vs Entry Threshold'); axes[1].legend()
plt.tight_layout(); plt.show()

best_thresh_sharpe = thresholds[np.argmax(sharpe_vals)]
best_thresh_pnl    = thresholds[np.argmax(pnl_vals)]
print(f"Best threshold by Sharpe: {best_thresh_sharpe:.2f} (Sharpe={max(sharpe_vals):.3f})")
print(f"Best threshold by PnL:    {best_thresh_pnl:.2f} (PnL={max(pnl_vals):.2f})")